In [ ]:
# --- Cell 1: Setup ---
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write('KGAT_9ff892943d6d52e13ca6e53a2786d62b')
!chmod 600 /root/.kaggle/access_token

Mounted at /content/drive


In [ ]:
# --- Cell 3: Download CarDD (Kaggle, YOLO-format version) ---
download_path = '/content/drive/MyDrive/capstone-vehicle-metadata/data/cardd_kaggle'
os.makedirs(download_path, exist_ok=True)

!kaggle datasets download -d gabrielfcarvalho/cardd-with-yolo-annotations-images-labels -p {download_path}

Dataset URL: https://www.kaggle.com/datasets/gabrielfcarvalho/cardd-with-yolo-annotations-images-labels
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
cardd-with-yolo-annotations-images-labels.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
# --- Cell 4: Extract to LOCAL disk (fast, small dataset ~4000 images) ---
import zipfile

zip_files = [f for f in os.listdir(download_path) if f.endswith('.zip')]
zip_path = os.path.join(download_path, zip_files[0])
local_extract_path = '/content/cardd_local'

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(local_extract_path)

print("Extraction complete.")
print(os.listdir(local_extract_path))

Extraction complete.
['test', 'val', 'train', 'data.yaml']


In [ ]:
# --- Cell 5: Inspect folder structure ---
for root, dirs, files in os.walk(local_extract_path):
    print(root, '->', len(files), 'files', '| sample:', files[:2])

/content/cardd_local -> 1 files | sample: ['data.yaml']
/content/cardd_local/test -> 0 files | sample: []
/content/cardd_local/test/images -> 374 files | sample: ['003027.jpg', '000330.jpg']
/content/cardd_local/test/labels -> 374 files | sample: ['000778.txt', '003477.txt']
/content/cardd_local/val -> 0 files | sample: []
/content/cardd_local/val/images -> 810 files | sample: ['000107.jpg', '000342.jpg']
/content/cardd_local/val/labels -> 810 files | sample: ['000311.txt', '001811.txt']
/content/cardd_local/train -> 0 files | sample: []
/content/cardd_local/train/images -> 2816 files | sample: ['003726.jpg', '002722.jpg']
/content/cardd_local/train/labels -> 2816 files | sample: ['003306.txt', '002134.txt']


In [ ]:
# --- Cell 6: Confirm YOLO label format (labels are .txt files, one per image) ---
# YOLO format: class_id x_center y_center width height (normalized 0-1)
sample_label_dir = os.path.join(local_extract_path, 'train', 'labels')  # adjust based on Cell 5 output
sample_labels = os.listdir(sample_label_dir)[:3]
for lbl in sample_labels:
    with open(os.path.join(sample_label_dir, lbl)) as f:
        print(lbl, ':', f.read())

003306.txt : 3 0.500000 0.598670 1.000000 0.802660

002134.txt : 0 0.166615 0.622249 0.137910 0.192594

001480.txt : 0 0.411450 0.061604 0.105960 0.121259
1 0.430005 0.531777 0.392770 0.808981



In [ ]:
# --- Cell 7: Build inventory (image path, split, label file path, damage classes present) ---
import pandas as pd
from PIL import Image
from tqdm import tqdm

class_names = ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat']

def parse_yolo_label(label_path):
    classes_present = []
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                class_id = int(line.split()[0])
                classes_present.append(class_names[class_id])
    return list(set(classes_present))

records = []
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(local_extract_path, split, 'images')
    lbl_dir = os.path.join(local_extract_path, split, 'labels')
    if not os.path.exists(img_dir):
        continue
    for fname in tqdm(os.listdir(img_dir), desc=f'Processing {split}'):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(img_dir, fname)
            lbl_path = os.path.join(lbl_dir, os.path.splitext(fname)[0] + '.txt')
            try:
                with Image.open(img_path) as img:
                    width, height = img.size
                corrupted = False
            except Exception:
                width, height = None, None
                corrupted = True
            damage_classes = parse_yolo_label(lbl_path)
            records.append({
                'file_path': img_path,
                'split': split,
                'width': width,
                'height': height,
                'damage_classes': damage_classes,
                'corrupted': corrupted
            })

df_cardd = pd.DataFrame(records)
print(f"Total images: {len(df_cardd)}")
print(f"Corrupted: {df_cardd['corrupted'].sum()}")
print(df_cardd['split'].value_counts())

Processing test: 100%|██████████| 374/374 [00:00<00:00, 515.30it/s]


Total images: 4000
Corrupted: 0
split
train    2816
val       810
test      374
Name: count, dtype: int64


In [ ]:
# --- Cell 8: Check damage class distribution ---
from collections import Counter

all_classes = [cls for classes in df_cardd['damage_classes'] for cls in classes]
print(Counter(all_classes))

# Images with no damage classes at all (label file empty or missing)
print(f"Images with no damage label: {(df_cardd['damage_classes'].apply(len) == 0).sum()}")

Counter({'scratch': 2121, 'dent': 1751, 'lamp broken': 693, 'glass shatter': 674, 'crack': 604, 'tire flat': 309})
Images with no damage label: 0


In [ ]:
# --- Cell 9: Check for duplicates (same pattern as CompCars) ---
import hashlib

def get_image_hash(filepath):
    try:
        with open(filepath, 'rb') as f:
            return hashlib.md5(f.read()).hexdigest()
    except Exception:
        return None

tqdm.pandas(desc='Hashing images')
df_cardd['image_hash'] = df_cardd['file_path'].progress_apply(get_image_hash)
print(f"Duplicate images: {df_cardd['image_hash'].duplicated().sum()}")

Hashing images: 100%|██████████| 4000/4000 [00:12<00:00, 324.69it/s]

Duplicate images: 0


In [ ]:
# --- Cell 10: Fix file paths to persistent Drive location before saving ---
df_cardd['file_path'] = df_cardd['file_path'].str.replace(
    local_extract_path,
    '/content/drive/MyDrive/capstone-vehicle-metadata/data/cardd_kaggle/extracted',
    regex=False
)

os.makedirs('/content/drive/MyDrive/capstone-vehicle-metadata/outputs', exist_ok=True)
df_cardd.to_csv('/content/drive/MyDrive/capstone-vehicle-metadata/outputs/cardd_inventory.csv', index=False)
print(f"Saved {len(df_cardd)} rows.")

Saved 4000 rows.


In [ ]:
# --- Cell 11: Copy the extracted local folder to Drive so paths in Cell 10 actually resolve ---
import shutil
shutil.copytree(
    local_extract_path,
    '/content/drive/MyDrive/capstone-vehicle-metadata/data/cardd_kaggle/extracted',
    dirs_exist_ok=True
)
print("Copied to Drive.")

Copied to Drive.


In [ ]:
for root, dirs, files in os.walk(local_extract_path):
    print(root, '->', len(files), 'files', '| sample:', files[:2])

/content/cardd_local -> 1 files | sample: ['data.yaml']
/content/cardd_local/test -> 0 files | sample: []
/content/cardd_local/test/images -> 374 files | sample: ['003027.jpg', '000330.jpg']
/content/cardd_local/test/labels -> 374 files | sample: ['000778.txt', '003477.txt']
/content/cardd_local/val -> 0 files | sample: []
/content/cardd_local/val/images -> 810 files | sample: ['000107.jpg', '000342.jpg']
/content/cardd_local/val/labels -> 810 files | sample: ['000311.txt', '001811.txt']
/content/cardd_local/train -> 0 files | sample: []
/content/cardd_local/train/images -> 2816 files | sample: ['003726.jpg', '002722.jpg']
/content/cardd_local/train/labels -> 2816 files | sample: ['003306.txt', '002134.txt']


In [ ]:
with open('/content/cardd_local/data.yaml') as f:
    print(f.read())


# COCO-like Car Damage Dataset converted to YOLO
train: /kaggle/input/cardd-with-yolo-annotations-images-labels/train/images
val: /kaggle/input/cardd-with-yolo-annotations-images-labels/val/images
test: /kaggle/input/cardd-with-yolo-annotations-images-labels/test/images

nc: 6
names: ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat']



In [ ]:
import pandas as pd
from PIL import Image
from tqdm import tqdm
import os

# Update this list based on what data.yaml actually shows in Step 4
class_names = ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat']

def parse_yolo_label(label_path):
    classes_present = []
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                if line.strip():
                    class_id = int(line.split()[0])
                    classes_present.append(class_names[class_id])
    return list(set(classes_present))

records = []
for split in ['train', 'val', 'test']:
    img_dir = f'/content/cardd_local/{split}/images'
    lbl_dir = f'/content/cardd_local/{split}/labels'
    for fname in tqdm(os.listdir(img_dir), desc=f'Processing {split}'):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(img_dir, fname)
            lbl_path = os.path.join(lbl_dir, os.path.splitext(fname)[0] + '.txt')
            try:
                with Image.open(img_path) as img:
                    width, height = img.size
                corrupted = False
            except Exception:
                width, height = None, None
                corrupted = True
            damage_classes = parse_yolo_label(lbl_path)
            records.append({
                'file_path': img_path,
                'split': split,
                'width': width,
                'height': height,
                'damage_classes': damage_classes,
                'corrupted': corrupted
            })

df_cardd = pd.DataFrame(records)
print(f"Total: {len(df_cardd)}")
print(f"Corrupted: {df_cardd['corrupted'].sum()}")
print(df_cardd['split'].value_counts())

Processing test: 100%|██████████| 374/374 [00:00<00:00, 4244.36it/s]

Total: 4000
Corrupted: 0
split
train    2816
val       810
test      374
Name: count, dtype: int64


In [ ]:
from collections import Counter

all_classes = [cls for classes in df_cardd['damage_classes'] for cls in classes]
print(Counter(all_classes))
print(f"Images with NO damage label: {(df_cardd['damage_classes'].apply(len) == 0).sum()}")

Counter({'scratch': 2121, 'dent': 1751, 'lamp broken': 693, 'glass shatter': 674, 'crack': 604, 'tire flat': 309})
Images with NO damage label: 0


In [ ]:
import hashlib

def get_image_hash(filepath):
    try:
        with open(filepath, 'rb') as f:
            return hashlib.md5(f.read()).hexdigest()
    except Exception:
        return None

tqdm.pandas(desc='Hashing')
df_cardd['image_hash'] = df_cardd['file_path'].progress_apply(get_image_hash)
print(f"Duplicates: {df_cardd['image_hash'].duplicated().sum()}")

Hashing: 100%|██████████| 4000/4000 [00:20<00:00, 199.14it/s]

Duplicates: 0


In [ ]:
# Copy to Drive first so paths resolve for teammates
import shutil
shutil.copytree('/content/cardd_local', '/content/drive/MyDrive/capstone-vehicle-metadata/data/cardd_kaggle/extracted', dirs_exist_ok=True)

df_cardd['file_path'] = df_cardd['file_path'].str.replace(
    '/content/cardd_local',
    '/content/drive/MyDrive/capstone-vehicle-metadata/data/cardd_kaggle/extracted',
    regex=False
)

os.makedirs('/content/drive/MyDrive/capstone-vehicle-metadata/outputs', exist_ok=True)
df_cardd.to_csv('/content/drive/MyDrive/capstone-vehicle-metadata/outputs/cardd_inventory.csv', index=False)
print(f"Saved {len(df_cardd)} rows.")

Saved 4000 rows.


In [ ]:
df_check = pd.read_csv('/content/drive/MyDrive/capstone-vehicle-metadata/outputs/cardd_inventory.csv')
print(df_check['split'].value_counts())
df_check.head(3)

split
train    2816
val       810
test      374
Name: count, dtype: int64


,file_path,split,width,height,damage_classes,corrupted,image_hash
0,/content/drive/MyDrive/capstone-vehicle-metada...,train,1000,562,['dent'],False,448ad361c942ff6f6fb2fff6f49e4eb1
1,/content/drive/MyDrive/capstone-vehicle-metada...,train,1000,668,"['scratch', 'crack']",False,81f14a322c8a7f8b306fa86cb41dd97a
2,/content/drive/MyDrive/capstone-vehicle-metada...,train,1000,787,"['dent', 'glass shatter']",False,52678ea17ef259da98763f9e4819f27c
